In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression  # <--- NUEVO: Modelo 1
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB

# Cargar el dataset original
df_original = pd.read_csv('Neo_Met_plants.csv')
Med_Plants = df_original.copy()

print("Dataset original cargado.")
print(f"Forma original: {Med_Plants.shape}")
print("\n✅ Todas las librerías importadas correctamente.")

Dataset original cargado.
Forma original: (1341, 20)

✅ Todas las librerías importadas correctamente.


In [2]:
# 1. Filtrar solo las filas donde Cereal sea 'barley' o 'wheat' para el target
df_wheat_barley = Med_Plants[Med_Plants['Cereal'].isin(['barley', 'wheat'])].copy()

print(f"Filas después del filtro (solo barley y wheat): {df_wheat_barley.shape[0]}")

# 2. Verificar cuántos hay de cada uno
print("\nConteo por tipo de cereal:")
print(df_wheat_barley['Cereal'].value_counts())

# 3. Crear la columna 'Cereal_encoded' (0 = barley, 1 = wheat)
df_wheat_barley['Cereal_encoded'] = (df_wheat_barley['Cereal'] == 'wheat').astype(int)

# Verificar que la codificación está correcta
print("\nVerificación de codificación:")
print(df_wheat_barley[['Cereal', 'Cereal_encoded']].head(10))

Filas después del filtro (solo barley y wheat): 1147

Conteo por tipo de cereal:
Cereal
barley    586
wheat     561
Name: count, dtype: int64

Verificación de codificación:
    Cereal  Cereal_encoded
2   barley               0
3   barley               0
4    wheat               1
5    wheat               1
6    wheat               1
7    wheat               1
8    wheat               1
9   barley               0
10  barley               0
11  barley               0


In [3]:
# 1. Codificar 'Chronological_Period_clean' como números, crear Período y Cuenca
le_periodo = LabelEncoder()
df_wheat_barley['Periodo_encoded'] = le_periodo.fit_transform(df_wheat_barley['Chronological_Period_clean'])

# 2. Codificar 'Mediterranean_Basin' como números
le_cuenca = LabelEncoder()
df_wheat_barley['Cuenca_encoded'] = le_cuenca.fit_transform(df_wheat_barley['Mediterranean_Basin'])

print("Nuevas columnas creadas:")
print(f"  - Periodo_encoded (valores únicos: {df_wheat_barley['Periodo_encoded'].nunique()})")
print(f"  - Cuenca_encoded (valores únicos: {df_wheat_barley['Cuenca_encoded'].nunique()})")

Nuevas columnas creadas:
  - Periodo_encoded (valores únicos: 8)
  - Cuenca_encoded (valores únicos: 3)


In [4]:
# 1. Definir las columnas predictoras, definir X e y
feature_columns = [
    'IRMS_d13C_Collagen',
    'd15N_Collagen',
    'Latitude_N',
    'Longitude_E',
    'Periodo_encoded',
    'Cuenca_encoded'
]

# Verificar que todas las columnas existen
missing_cols = [col for col in feature_columns if col not in df_wheat_barley.columns]
if missing_cols:
    print(f"⚠️ ATENCIÓN: Estas columnas no existen: {missing_cols}")
else:
    print("✓ Todas las columnas predictoras están presentes.")

# 2. Definir X e y
X = df_wheat_barley[feature_columns]
y = df_wheat_barley['Cereal_encoded']

print(f"\nVariables predictoras (X): {X.shape}")
print(f"Variable objetivo (y): {y.shape}")
print(f"\nDistribución de clases en 'y':")
print(y.value_counts())

✓ Todas las columnas predictoras están presentes.

Variables predictoras (X): (1147, 6)
Variable objetivo (y): (1147,)

Distribución de clases en 'y':
Cereal_encoded
0    586
1    561
Name: count, dtype: int64


In [5]:
# Dividir los datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño entrenamiento: {X_train.shape[0]} muestras")
print(f"Tamaño prueba: {X_test.shape[0]} muestras")

Tamaño entrenamiento: 917 muestras
Tamaño prueba: 230 muestras


In [8]:
# Escalar datos (opcional, Naive Bayes funciona con distribuciones de probabilidad)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
# Naive Bayes - asume independencia de características
model = GaussianNB()
model.fit(X_train_scaled, y_train)

#print("Modelo Naive Bayes entrenado correctamente.")

,priors,None
,var_smoothing,1e-09


In [12]:
# Predecir sobre el conjunto de prueba
y_pred = model.predict(X_test_scaled)

# Métricas de rendimiento
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud (Accuracy): {accuracy:.4f}")

print("\n--- Informe de clasificación ---")
print(classification_report(y_test, y_pred, target_names=['cebada (0)', 'trigo (1)']))

print("\n--- Matriz de Confusión ---")
print(confusion_matrix(y_test, y_pred))

# Comparar primeras predicciones
print("\n--- Comparación (primeros 10 casos de prueba) ---")
comparison = pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicción': y_pred[:10],
    'Correcta?': y_test.values[:10] == y_pred[:10]
})
print(comparison)

Exactitud (Accuracy): 0.6217

--- Informe de clasificación ---
              precision    recall  f1-score   support

  cebada (0)       0.67      0.52      0.58       118
   trigo (1)       0.59      0.73      0.65       112

    accuracy                           0.62       230
   macro avg       0.63      0.62      0.62       230
weighted avg       0.63      0.62      0.62       230


--- Matriz de Confusión ---
[[61 57]
 [30 82]]

--- Comparación (primeros 10 casos de prueba) ---
   Real  Predicción  Correcta?
0     0           1      False
1     1           0      False
2     0           1      False
3     1           1       True
4     0           1      False
5     0           0       True
6     1           1       True
7     0           0       True
8     1           1       True
9     0           0       True


In [15]:
# Para Naive Bayes, ver medias y varianzas por clase
print("\n--- Parámetros del modelo Naive Bayes ---")
print("Medias por clase:")
print(pd.DataFrame(model.theta_, columns=feature_columns, index=['cebada', 'trigo']))

# Nota: usamos 'var_' en lugar de 'sigma_' (varianza, no desviación estándar)
print("\nVarianzas por clase:")
print(pd.DataFrame(model.var_, columns=feature_columns, index=['cebada', 'trigo']))


--- Parámetros del modelo Naive Bayes ---
Medias por clase:
        IRMS_d13C_Collagen  d15N_Collagen  Latitude_N  Longitude_E  \
cebada           -0.115176       0.067188   -0.176321    -0.043332   
trigo             0.120050      -0.070031    0.183782     0.045166   

        Periodo_encoded  Cuenca_encoded  
cebada         0.149443        0.092357  
trigo         -0.155767       -0.096265  

Varianzas por clase:
        IRMS_d13C_Collagen  d15N_Collagen  Latitude_N  Longitude_E  \
cebada            0.874170       1.368472    1.257290     1.291383   
trigo             1.102916       0.606326    0.665642     0.692290   

        Periodo_encoded  Cuenca_encoded  
cebada         0.960595        1.058325  
trigo          0.993531        0.921049  


#### Explicación de Naive Bayes (Modelo 4) ¿En qué consiste?  

Naive Bayes es un algoritmo de clasificación basado en el Teorema de Bayes. Su nombre viene de la suposición "naive" (ingenua) de que todas las características son independientes entre sí.

En este caso significa que el algoritmo asume que el valor de d15N_Collagen no tiene relación con Latitude_N, que Longitude_E es independiente de Periodo_encoded, etc. Aunque esto raramente es cierto en la realidad, el modelo funciona sorprendentemente bien.

##### ¿Cómo funciona?
El algoritmo calcula la probabilidad de que una muestra pertenezca a cada clase (cebada o trigo) usando la probabilidad de cada característica por separado. Luego, asigna la muestra a la clase con mayor probabilidad.

Ejemplo simplificado: Imagina que quieres clasificar una muestra con d15N_Collagen = 8.5 y Latitude_N = 40.2. Naive Bayes calcula:

P(cebada) × P(d15N=8.5 | cebada) × P(Latitude=40.2 | cebada)

P(trigo) × P(d15N=8.5 | trigo) × P(Latitude=40.2 | trigo)

Y elige el valor más alto.

##### Ventajas:
Muy rápido de entrenar y predecir

Funciona bien con pocos datos

Buen baseline para comparar con modelos más complejos

Maneja bien características numéricas (con GaussianNB)

##### Desventajas:
La suposición de independencia es casi siempre falsa

Puede ser menos preciso que modelos como Random Forest o SVM

🎯 ¿Por qué es adecuado para la hipótesis 4?
No conozco tu hipótesis 4 específica, pero supongamos que es algo como:

"Las variables isotópicas y geográficas permiten clasificar el tipo de cereal (cebada vs trigo) independientemente unas de otras."

Naive Bayes es adecuado para esta hipótesis precisamente porque asume independencia. Si la hipótesis sostiene que las variables actúan de forma independiente, entonces Naive Bayes será un modelo apropiado. Además, es un excelente modelo de referencia para ver si relaciones más complejas (como las que captura Random Forest) realmente aportan mejora significativa.